<div style="display: flex; justify-content: flex-start; align-items: center;">
    <a href="https://colab.research.google.com/github/msfasha/307307-BI-Methods-Generative-AI/blob/main/20261/Module%206%20-%20Introduction%20to%20CrewAI/CrewAI_Playground_lab.ipynb" target="_blank">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 25px;">
    </a>
</div>

<div style="background-color:#F5EDF4; border-top:3px solid #A83296; padding:16px; border-radius:6px">
<h1 style="color:#493347; margin:0">CrewAI Playground</h1>
<p style="color:#655263; margin:4px 0 0 0">Agentic AI — University of Petra</p>
</div>

**A short, hands-on tour of CrewAI.** Eight code blocks, one idea each. Every block ends with a **Try it** box: change one thing, run again, and watch how the agent changes. That is the whole point of this notebook: **play**.

We work for **Petra Bean**, a small (fictional) neighbourhood coffee shop.

| Block | Idea |
|---|---|
| 1 | Setup: connect to the model |
| 2 | An **Agent**, a **Task** and a **Crew** |
| 3 | Change the **role**: same question, different persona |
| 4 | Change the **task**: control what the answer looks like |
| 5 | Reuse the crew with **inputs** |
| 6 | A **team** of two agents |
| 7 | Give an agent a **tool** |
| 8 | Ask for **structured output** |

## Block 1 — Setup
Install CrewAI and give it your Gemini key. In Colab, store the key as `GOOGLE_API_KEY` in the Secrets panel (key icon, left). Elsewhere you will be asked to paste it. The install takes a minute or two.

In [ ]:
# Lines starting with "!" are shell commands. Remove the "!" in a terminal.
!pip install -q "crewai[google-genai]"

In [ ]:
import os, getpass
from crewai import Agent, Task, Crew, LLM

# Read the key from Colab Secrets, otherwise ask for it.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
os.environ["OTEL_SDK_DISABLED"] = "true"        # no usage reporting
os.environ["CREWAI_TRACING_ENABLED"] = "false"

# The model every agent will use. If you get "model not found", change only this line.
llm = LLM(model="gemini/gemini-3.6-flash")
print(llm.call("Say hello to a coffee shop owner in one short sentence."))

## Block 2 — Agent, Task, Crew
Three building blocks, like a small company:
- **Agent** = an employee. It has a `role` (who), a `goal` (what it wants) and a `backstory` (how it behaves).
- **Task** = a job. A `description` (what to do) and an `expected_output` (what "done" looks like).
- **Crew** = the team that runs the tasks. `kickoff()` is the "go" button: nothing is sent to the model until you call it.

In [ ]:
analyst = Agent(
    role="Cafe Business Analyst",
    goal="Give small cafe owners clear, honest opinions",
    backstory="You have ten years of experience advising independent coffee shops. You are practical and say when you are unsure.",
    llm=llm,
)

task = Task(
    description="Petra Bean is thinking about selling bottled cold brew coffee. Is it a good idea?",
    expected_output="An answer of at most 60 words.",
    agent=analyst,
)

crew = Crew(agents=[analyst], tasks=[task])
result = crew.kickoff()
print(result.raw)   # the final answer is stored in result.raw

**Try it:** change the `description` to another idea (a loyalty card, a second branch, selling cakes) and run again.

## Block 3 — Change the role, change the agent
The `role`, `goal` and `backstory` are pasted into the model's instructions. So **changing the persona changes the answer**, even though the question is identical. Here one loop runs the same task with three different agents.

In [ ]:
personas = {
    "Cautious accountant": ("Cautious Accountant", "Protect the shop's money", "You hate risk and always think about what could go wrong."),
    "Enthusiastic barista": ("Enthusiastic Barista", "Make customers happy", "You love coffee, love new drinks, and get excited about new ideas."),
    "Marketing expert": ("Marketing Expert", "Attract new customers", "You think in terms of branding, social media and what makes people talk."),
}

for name, (role, goal, backstory) in personas.items():
    agent = Agent(role=role, goal=goal, backstory=backstory, llm=llm)
    t = Task(
        description="Petra Bean is thinking about selling bottled cold brew coffee. Give your opinion.",
        expected_output="An answer of at most 50 words.",
        agent=agent,
    )
    print(f"--- {name} ---")
    print(Crew(agents=[agent], tasks=[t]).kickoff().raw, "\n")

**Try it:** add a fourth persona (a grumpy regular customer? a health inspector?). Or keep one role and change only the `backstory` ("you only care about the environment") and compare.

## Block 4 — Change the task, change the answer
The `expected_output` is the strongest lever you have. Same agent, same question; only the definition of "done" changes.

In [ ]:
for expected in [
    "One sentence of at most 15 words.",
    "Exactly 3 bullet points: one benefit, one risk, one next step.",
    "A short poem of 4 lines.",
]:
    t = Task(
        description="Petra Bean is thinking about selling bottled cold brew coffee. Is it a good idea?",
        expected_output=expected,
        agent=analyst,
    )
    print(f"--- {expected} ---")
    print(Crew(agents=[analyst], tasks=[t]).kickoff().raw, "\n")

**Try it:** write your own `expected_output`, for example "A reply a 10-year-old would understand" or "A table with two columns".

## Block 5 — Inputs: one crew, many questions
Put `{placeholders}` in the task and fill them at run time with `inputs={...}`. You build the team once and reuse it.

In [ ]:
task = Task(
    description="Petra Bean is thinking about selling {idea}. Is it a good idea for a {season} launch?",
    expected_output="At most 40 words.",
    agent=analyst,
)
crew = Crew(agents=[analyst], tasks=[task])

for idea, season in [("iced lattes", "summer"), ("hot chocolate", "winter"), ("fresh pastries", "spring")]:
    print(f"--- {idea} / {season} ---")
    print(crew.kickoff(inputs={"idea": idea, "season": season}).raw, "\n")

**Try it:** add your own `{placeholder}` (for example `{audience}`) and a new value for it in `inputs`.

## Block 6 — A team: two agents, one after the other
Real work needs different skills. A **researcher** collects facts and a **writer** turns them into something for the owner. In a `Process.sequential` crew, tasks run in order and each task sees the **output of the one before it** (via `context`).

In [ ]:
from crewai import Process

researcher = Agent(
    role="Market Researcher",
    goal="List accurate, relevant facts",
    backstory="You are thorough, you stick to facts and you never exaggerate.",
    llm=llm,
)
writer = Agent(
    role="Friendly Writer",
    goal="Turn facts into a short, warm note for a busy shop owner",
    backstory="You write simply and kindly. No jargon.",
    llm=llm,
)

research = Task(
    description="List 4 facts a coffee shop owner should know before selling {idea}.",
    expected_output="4 short bullet points.",
    agent=researcher,
)
note = Task(
    description="Using the research, write a short note to the owner of Petra Bean about {idea}.",
    expected_output="A note of at most 80 words that ends with one recommendation.",
    agent=writer,
    context=[research],          # this task receives the researcher's output
)

team = Crew(agents=[researcher, writer], tasks=[research, note], process=Process.sequential)
print(team.kickoff(inputs={"idea": "bottled cold brew coffee"}).raw)

**Try it:** change only the writer's `backstory` ("You write like a pirate", "You write like a news reporter") and watch the note change while the facts stay the same.

## Block 7 — Tools: let the agent use real code
Language models are fluent but **unreliable at exact arithmetic**. A **tool** is an ordinary Python function the agent may call. The `@tool` decorator registers it, and **the docstring is the manual** the model reads to decide when and how to use it, so write it clearly.

In [ ]:
from crewai.tools import tool

@tool("Profit calculator")
def profit(price: float, unit_cost: float, units: int) -> str:
    """Calculate total profit for a product.
    Inputs: price (selling price per unit), unit_cost (cost per unit), units (number sold).
    Returns the total profit."""
    return f"Total profit = {(price - unit_cost) * units:.2f}"

accountant = Agent(
    role="Careful Accountant",
    goal="Answer money questions exactly, never guessing a number",
    backstory="You always use the calculator for arithmetic.",
    llm=llm,
    tools=[profit],              # the only new line compared with a normal agent
    verbose=True,                # watch the log: you will see the tool being called
)
t = Task(
    description="Petra Bean sells 1,240 bottles of cold brew at 4.50 each, costing 1.30 each. What is the total profit?",
    expected_output="One sentence with the profit.",
    agent=accountant,
)
print(Crew(agents=[accountant], tasks=[t]).kickoff().raw)   # correct answer: 3968.00

**Try it:** delete the "Inputs:" line from the docstring and run again. Does the agent still call the tool correctly? Then remove `tools=[profit]` and see whether the number is still right.

## Block 8 — Structured output
Free text is hard to use in a program. If a later step needs a *field* (a verdict, a score), ask for a **Pydantic model** with `output_pydantic`. The answer then comes back as an object with exact fields.

In [ ]:
from typing import List, Literal
from pydantic import BaseModel

class Verdict(BaseModel):
    decision: Literal["go", "maybe", "no-go"]
    reasons: List[str]

t = Task(
    description="Should Petra Bean sell {idea}?",
    expected_output="A decision and up to 3 short reasons.",
    agent=analyst,
    output_pydantic=Verdict,
)
out = Crew(agents=[analyst], tasks=[t]).kickoff(inputs={"idea": "bottled cold brew coffee"})

print(out.pydantic.decision)     # always exactly "go", "maybe" or "no-go"
print(out.pydantic.reasons)      # a real Python list

**Try it:** add a field such as `confidence: int` or `next_step: str` to `Verdict` and run again.

---
## What next?
- Build your own agent for a business you know and give it a persona.
- Move on to the longer lab, *Building AI Teams* (`CrewAI_Intro_lab.ipynb`): data, knowledge, memory, guardrails, flows.